(Dev) If you are re-running this, only run the following cells: 
1. os.chdir("../")
2. load_dotenv()
3. embeddings_init()
4. p_client = Pinecone(...)
5. .Index()

In [1]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\ASUS\\Desktop\\Projects\\Medical_RAG\\Medical-RAG-Chatbot'

In [3]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\ASUS\anaconda3\envs\medical-rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_pdfs(data_dir):
    loader = DirectoryLoader(
        data_dir,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documets = loader.load()
    return documets

In [ ]:
extracted_data = load_pdfs("./data")

In [7]:
extracted_data[0]

Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book.pdf', 'total_pages': 637, 'page': 0, 'page_label': '1'}, page_content='')

In [6]:
len(extracted_data)

637

In [4]:
from typing import List
from langchain.schema import Document

def filter(docs: List[Document]) -> List[Document]:
    docs_filtered: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        docs_filtered.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return docs_filtered

In [12]:
docs_filtered = filter(extracted_data)
print(docs_filtered[10])

page_content='Rhonda Cloos, R.N.
Medical Writer
Austin, TX
Gloria Cooksey, C.N.E
Medical Writer
Sacramento, CA
Amy Cooper, M.A., M.S.I.
Medical Writer
Vermillion, SD
David A. Cramer, M.D.
Medical Writer
Chicago, IL
Esther Csapo Rastega, R.N., B.S.N.
Medical Writer
Holbrook, MA
Arnold Cua, M.D.
Physician
Brooklyn, NY
Tish Davidson, A.M.
Medical Writer
Fremont, California
Dominic De Bellis, Ph.D.
Medical Writer/Editor
Mahopac, NY
Lori De Milto
Medical Writer
Sicklerville, NJ
Robert S. Dinsmoor
Medical Writer
South Hamilton, MA
Stephanie Dionne, B.S.
Medical Writer
Ann Arbor, MI
Martin W. Dodge, Ph.D.
Technical Writer/Editor
Centinela Hospital and Medical
Center
Inglewood, CA
David Doermann
Medical Writer
Salt Lake City, UT
Stefanie B. N. Dugan, M.S.
Genetic Counselor
Milwaukee, WI
Doug Dupler, M.A.
Science Writer
Boulder, CO
Julie A. Gelderloos
Biomedical Writer
Playa del Rey, CA
Gary Gilles, M.A.
Medical Writer
Wauconda, IL
Harry W. Golden
Medical Writer
Shoreline Medical Writers
Old Ly

In [5]:
# Chunking
def text_splitter(docs_filtered):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    text_chunk = text_splitter.split_documents(docs_filtered)
    return text_chunk

In [16]:
text_chunk = text_splitter(docs_filtered)

In [6]:
from langchain.embeddings import HuggingFaceEmbeddings

def embeddings_init():
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embeddings = embeddings_init()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_27876\3408261459.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)


In [19]:
vector = embeddings.embed_query("Hello world")
len(vector)

384

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [3]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [4]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

p_client = Pinecone(api_key=pinecone_api_key)

In [5]:
from pinecone import ServerlessSpec

index_name = "medical-rag-chatbot"

if not p_client.has_index(index_name):
    p_client.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )

index = p_client.Index(index_name)

c:\Users\ASUS\anaconda3\envs\medical-rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from langchain_pinecone import PineconeVectorStore

In [ ]:
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embeddings,
    index_name=index_name
)

In [14]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

### Adding more data to our existing Pinecone Index

In [28]:
test123 = Document(
    page_content="Alan Fernandes is a 2nd year AI&DS student studying in Ramaiah Institute of Technology.",
    metadata={"source":"MSRIT"}
)

In [29]:
docsearch.add_documents(documents=[test123])

['3a702f39-d81f-4787-9682-fda018bfced9']

In [16]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [31]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='109061a9-2f52-4ce6-8b22-5de7d5484d39', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='9cca4718-b1e0-4f8d-9e50-42dbbf59d26a', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='cb669509-6cff-4e12-96f7-95f462777d26', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged wi

In [7]:
from langchain_groq import ChatGroq
chatModel = ChatGroq(model="llama-3.3-70b-versatile",temperature=0.4)

In [8]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [10]:
system_prompt = (
    "You are a Medical assistant that answers user's questions"
    "Use the following pieces of retrieved context to answer"
    "the question better. If the answer is not known to you, claim"
    "that you don't know. Use three sentences at max and keep the answer"
    "concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
)

In [17]:
question_answer_chain = create_stuff_documents_chain(chatModel,prompt)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [22]:
response = rag_chain.invoke({"input":"What is the treatment for Acne?"})
print(response["answer"])

The treatment for acne depends on its severity, but for mild cases, topical drugs such as tretinoin, benzoyl peroxide, and salicylic acid are used to reduce comedone formation. For inflammatory acne, topical antibiotics may be added to the treatment regimen. Improvement is usually seen in two to four weeks, but more severe cases may require oral medications like isotretinoin, which can take two or more months to show improvement.


### It works!